前情提要：当项目进行到 pyscenic grn 的准备工作时，尝试使用果蝇的 TF 列表基于同源关系获取东方蜜蜂的 TF 列表。

此时，注意到果蝇-东方蜜蜂的 OrthoFinder 结果文件(tsv格式)中，果蝇使用的是 GTF 中的 gene_id，但蜜蜂使用的是 protein_id。

根据 OrthoFinder 的原理，可以推测，其原因是输入 OrthoFinder 中的蜜蜂 fasta，其 header 对应的是 protein_id 而非 gene_id。

那么，这将带来一个问题：如果蜜蜂 header 对应的是 protein_id 而非 gene_id，可能 OrthoFinder 之前的 primary_transcript 步骤就无法有效地去除 fasta 中属于同一基因的较短的转录本。这可能进一步导致同一基因的多个 isoform 被分到不同 Orthogroup。

In [4]:
%%bash

# 运行primary_transcript之前的蜜蜂fasta：
cp "/home/liuzhiyu/Projects/neo_caste/Find_Ortholog/proteomes/GCF_029169275.1_AcerK_1.0_protein.faa" "./orthofinder_output/GCF_029169275.1_AcerK_1.0_protein.faa"
# primary_transcript处理后的蜜蜂fasta：
cp "/home/liuzhiyu/Projects/neo_caste/Find_Ortholog/primary_transcripts/Apis_cerana.faa" "./orthofinder_output/Apis_cerana.faa"

du -sh "./orthofinder_output/GCF_029169275.1_AcerK_1.0_protein.faa"
du -sh "./orthofinder_output/Apis_cerana.faa"

25M	./orthofinder_output/GCF_029169275.1_AcerK_1.0_protein.faa
8.2M	./orthofinder_output/Apis_cerana.faa


根据文件的大小，可以确定 primary_transcript 确实对同一基因的短转录本进行了清理，但目前无法确定是否清理干净，需要验证。

我的想法是：先从蜜蜂 GTF 中提取各 gene_id 及其对应的各转录本 protein_id，以表格形式存储。然后获取 primary_transcript 处理后的蜜蜂 fasta 中的 protein_id，确定前者单个 gene_id 是否仅对应后者单个 protein_id，如果是，证明 primary_transcript 的处理可能是干净的。

In [2]:
%%bash

ln -sf "/home/liuzhiyu/Projects/neo_caste/fastq2matrix/Acer/ref/GCF_029169275.1_fixed.filtered.gtf" "./gtf/Apis_cerana.gtf"
gppy txinfo -g "./gtf/Apis_cerana.gtf" > "./gtf/Apis_cerana.tsv"

In [1]:
import pandas as pd

Acer_gtf = pd.read_csv("./gtf/Apis_cerana.tsv", sep="\t")
Acer_gid_pid = Acer_gtf[["gene", "protein_id"]].drop_duplicates()
Acer_gid_pid

/tmp/ipykernel_47999/2772417620.py:3: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  Acer_gtf = pd.read_csv("./gtf/Apis_cerana.tsv", sep="\t")


,gene,protein_id
0,LOC107992807,XP_016904367.1
1,LOC107992807,XP_016904368.1
2,LOC107992805,XP_061938525.1
3,LOC107992805,XP_061938550.1
4,LOC107992805,XP_061938560.1
...,...,...
34710,ND4,YP_003735177.1
34711,ND4L,YP_003735178.1
34712,ND6,YP_003735179.1
34713,CYTB,YP_003735180.1


In [2]:
from pyfaidx import Fasta

Acer_faa = Fasta("./orthofinder_output/Apis_cerana.faa")
Acer_faa_header_set = set(Acer_faa.keys())
Acer_faa_header_set

{'XP_028520122.1',
 'XP_016920836.1',
 'XP_061934535.1',
 'XP_061936518.1',
 'XP_061932386.1',
 'XP_061942643.1',
 'XP_016918557.1',
 'XP_016918395.1',
 'XP_061939137.1',
 'XP_061941369.1',
 'XP_016914972.1',
 'XP_061941732.1',
 'XP_016914682.1',
 'XP_016904308.2',
 'XP_016919393.2',
 'XP_016910682.1',
 'XP_016914794.2',
 'XP_016906944.2',
 'XP_016919530.1',
 'XP_016915862.2',
 'XP_028520668.2',
 'XP_061939225.1',
 'XP_016909883.2',
 'XP_061937709.1',
 'XP_016906850.1',
 'XP_016911464.1',
 'XP_016907082.2',
 'XP_016908701.1',
 'XP_016909757.1',
 'XP_016915127.1',
 'XP_061931859.1',
 'XP_016918925.1',
 'XP_061936360.1',
 'XP_061939598.1',
 'XP_061942978.1',
 'XP_016903917.1',
 'XP_061931080.1',
 'XP_016909087.2',
 'XP_061942584.1',
 'XP_016915185.1',
 'XP_016919701.1',
 'XP_016922417.1',
 'XP_061935835.1',
 'XP_061937700.1',
 'XP_016918774.2',
 'XP_016903957.1',
 'XP_061936842.1',
 'XP_028525710.1',
 'XP_016907115.1',
 'XP_016906527.1',
 'XP_061929607.1',
 'XP_028524099.2',
 'XP_0619338

In [3]:
Acer_gid_pid_infaa = Acer_gid_pid[Acer_gid_pid["protein_id"].isin(Acer_faa_header_set)]
Acer_gid_pid_infaa

,gene,protein_id
0,LOC107992807,XP_016904367.1
1,LOC107992807,XP_016904368.1
9,LOC107992806,XP_061938570.1
11,LOC107992868,XP_028520126.1
14,LOC107992792,XP_016904346.1
...,...,...
34710,ND4,YP_003735177.1
34711,ND4L,YP_003735178.1
34712,ND6,YP_003735179.1
34713,CYTB,YP_003735180.1


In [7]:
Acer_gid_pid_infaa["gene"].value_counts()

LOC107998865    18
LOC108001457    18
LOC108001506    17
LOC108001085    17
LOC108000242    15
                ..
LOC107997847     1
LOC107997846     1
LOC107997848     1
LOC107997849     1
ND1              1
Name: gene, Length: 9717, dtype: int64

↑ 综上，可见 primary_transcript 处理后的东方蜜蜂 fasta 中，同一基因有若干转录本，primary_transcript 并未保留单一的最长转录本，因此之前的 OrthoFinder 实践是不正确的。